# 00 · 데이터셋 준비 및 가공

Phase 2 — `KorQuAD/squad_kor_v1` 원본을 로드하고 RAG 평가용 골든셋을 추출한다.

## 진행 단계
1. **원본 로드** — `datasets`로 `KorQuAD/squad_kor_v1` 로드 _(현재 단계)_
2. 구조 파악 및 가공 — `title` 기준 context 그룹핑 / dedup → `data/docs/`
3. 골든셋 추출 — `(question, context, answers)` 50개 샘플링 → `data/golden_set.json`

## 1. 원본 로드

In [1]:
from datasets import load_dataset

dataset = load_dataset("KorQuAD/squad_kor_v1")
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 60407
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 5774
    })
})

In [2]:
# split 별 샘플 수 및 첫 번째 예시 확인
train = dataset["train"]
print(f"train: {len(dataset['train'])} / validation: {len(dataset['validation'])}")
print("features:", train.features)
train[0]

train: 60407 / validation: 5774
features: {'id': Value('string'), 'title': Value('string'), 'context': Value('string'), 'question': Value('string'), 'answers': {'text': List(Value('string')), 'answer_start': List(Value('int32'))}}


{'id': '6566495-0-0',
 'title': '파우스트_서곡',
 'context': '1839년 바그너는 괴테의 파우스트을 처음 읽고 그 내용에 마음이 끌려 이를 소재로 해서 하나의 교향곡을 쓰려는 뜻을 갖는다. 이 시기 바그너는 1838년에 빛 독촉으로 산전수전을 다 걲은 상황이라 좌절과 실망에 가득했으며 메피스토펠레스를 만나는 파우스트의 심경에 공감했다고 한다. 또한 파리에서 아브네크의 지휘로 파리 음악원 관현악단이 연주하는 베토벤의 교향곡 9번을 듣고 깊은 감명을 받았는데, 이것이 이듬해 1월에 파우스트의 서곡으로 쓰여진 이 작품에 조금이라도 영향을 끼쳤으리라는 것은 의심할 여지가 없다. 여기의 라단조 조성의 경우에도 그의 전기에 적혀 있는 것처럼 단순한 정신적 피로나 실의가 반영된 것이 아니라 베토벤의 합창교향곡 조성의 영향을 받은 것을 볼 수 있다. 그렇게 교향곡 작곡을 1839년부터 40년에 걸쳐 파리에서 착수했으나 1악장을 쓴 뒤에 중단했다. 또한 작품의 완성과 동시에 그는 이 서곡(1악장)을 파리 음악원의 연주회에서 연주할 파트보까지 준비하였으나, 실제로는 이루어지지는 않았다. 결국 초연은 4년 반이 지난 후에 드레스덴에서 연주되었고 재연도 이루어졌지만, 이후에 그대로 방치되고 말았다. 그 사이에 그는 리엔치와 방황하는 네덜란드인을 완성하고 탄호이저에도 착수하는 등 분주한 시간을 보냈는데, 그런 바쁜 생활이 이 곡을 잊게 한 것이 아닌가 하는 의견도 있다.',
 'question': '바그너는 괴테의 파우스트를 읽고 무엇을 쓰고자 했는가?',
 'answers': {'text': ['교향곡'], 'answer_start': [54]}}

## 2. 구조 파악 및 가공

`squad_kor_v1`은 `title` 단위로 여러 paragraph(`context`)가 묶여 있고, **같은 context가 여러 QA 쌍에서 중복 등장**한다.
RAG 검색 대상 코퍼스를 만들기 위해 `title` 기준으로 그룹핑하고 중복 context를 제거(dedup)한 뒤, title별 텍스트 파일로 `data/docs/`에 저장한다.

In [3]:
from collections import defaultdict

# title 기준 그룹핑 + context dedup (첫 등장 순서 보존 → 재현성 확보)
docs: dict[str, list[str]] = defaultdict(list)
_seen: dict[str, set[str]] = defaultdict(set)

for row in train:
    title, context = row["title"], row["context"]
    if context not in _seen[title]:
        _seen[title].add(context)
        docs[title].append(context)

print(f"문서(title) 수: {len(docs)}")
print(f"고유 context 수: {sum(len(v) for v in docs.values())}")

문서(title) 수: 1417
고유 context 수: 9621


In [4]:
import statistics

# 가공 결과 구조 통계
sizes = [len(v) for v in docs.values()]
total_ctx = sum(sizes)
dup_ratio = 1 - total_ctx / len(train)

print(f"총 문서(title): {len(docs)}")
print(f"고유 context : {total_ctx}  (원본 {len(train)}행 중 중복 {dup_ratio:.1%} 제거)")
print(f"title당 context 수 — min {min(sizes)} / median {int(statistics.median(sizes))} / max {max(sizes)}")

총 문서(title): 1417
고유 context : 9621  (원본 60407행 중 중복 84.1% 제거)
title당 context 수 — min 1 / median 4 / max 144


In [5]:
import re
from pathlib import Path

DOCS_DIR = Path("../data/docs")
DOCS_DIR.mkdir(parents=True, exist_ok=True)


def slugify(title: str) -> str:
    """파일명 안전화: 파일시스템 금지 문자 치환, 앞뒤 공백/점 제거."""
    s = re.sub(r'[<>:"/\\|?*\x00-\x1f]', "_", title).strip(" .")
    return s or "untitled"


# 같은 title의 context들을 빈 줄로 이어 하나의 문서로 저장
for title, contexts in docs.items():
    path = DOCS_DIR / f"{slugify(title)}.txt"
    path.write_text("\n\n".join(contexts), encoding="utf-8")

print(f"{len(docs)}개 문서를 {DOCS_DIR}/ 에 저장했습니다.")

1417개 문서를 ../data/docs/ 에 저장했습니다.


In [6]:
# 저장 결과 검증 — 파일 수 일치 확인 + 예시 미리보기
saved = sorted(DOCS_DIR.glob("*.txt"))
assert len(saved) == len(docs), f"파일 수 불일치: {len(saved)} != {len(docs)}"
print(f"저장된 파일: {len(saved)}개\n")

sample = saved[0]
print(f"예시 파일: {sample.name}")
print("-" * 60)
print(sample.read_text(encoding="utf-8")[:300], "...")

저장된 파일: 1417개

예시 파일: 12국_목록.txt
------------------------------------------------------------
각 나라에 하나씩 존재하는 최고위 신수. 왕기를 의지하여 자신의 주인을 선택한다. 그 뒤 왕과 함께 나라로 돌아가 신하가 되어 곁에서 왕을 돕는다. 기린의 성격은 기본적으로 자국민의 기질을 기준으로, 민의가 구체화 한 것이라고 여겨지고 있다. 성정은 인이며 자비심이 깊다. 싸움을 싫어하며 피의 부정으로 병이 들기도 한다. 자신의 주인 이외에는 결코 머리를 숙이지 않으며 다른 사람에게 머리를 숙이는 것은 절대로 불가능하다. 또한 기린 본래의 힘은 짐승으로 변했을 때 그 뿔에서 나온다고 여겨지며, 사람의 모습일 때는 뿔에 해당하는 이마 ...


## 3. 골든셋 추출

`(question, context, answers)` 삼중쌍에서 평가용 골든셋 50개를 샘플링한다.
- **필터 조건**: context 길이 100자 이상, answer 비어있지 않은 것
- **RAGAS 입력 형식**으로 변환 후 `data/golden_set.json`에 저장
- `seed` 고정으로 재현 가능한 샘플링 (Phase 4·5에서 동일 골든셋 재사용)

In [7]:
import random

GOLDEN_SIZE = 50
SEED = 42


def is_valid(row) -> bool:
    """필터: context 100자 이상 + answer 비어있지 않음."""
    answers = row["answers"]["text"]
    return len(row["context"]) >= 100 and len(answers) > 0 and answers[0].strip() != ""


# 조건 통과 후보 인덱스 수집 → seed 고정 샘플링
candidates = [i for i in range(len(train)) if is_valid(train[i])]
random.seed(SEED)
picked = random.sample(candidates, GOLDEN_SIZE)

print(f"필터 통과 후보: {len(candidates)} / 전체 {len(train)}")
print(f"샘플링: {len(picked)}개 (seed={SEED})")

필터 통과 후보: 60407 / 전체 60407
샘플링: 50개 (seed=42)


In [8]:
# RAGAS 입력 형식으로 변환
#   { "question": ..., "ground_truth": ..., "reference_contexts": [...] }
golden_set = [
    {
        "question": train[i]["question"],
        "ground_truth": train[i]["answers"]["text"][0],
        "reference_contexts": [train[i]["context"]],
    }
    for i in picked
]

# 예시 1개 확인
import json

print(json.dumps(golden_set[0], ensure_ascii=False, indent=2)[:400])

{
  "question": "구식 군인들의 월급인 쌀에 모래와 돌멩이가 들어가있던 사건을 말미암아 일어난 사태의 이름은?",
  "ground_truth": "임오군란",
  "reference_contexts": [
    "1882년 6월 민영익의 귀국 권고로 일시 귀국했다가 다시 되돌아갔다. 7월 23일 한성부에서 구식 군인들의 월급으로 주는 쌀에 모래와 돌멩이 및 썩은 쌀을 주자 여기에 반발한 구식 군인들에 의해 임오군란이 일어났는데, 그는 임오군란 사태 당시 행동을 삼가고 사태의 추이를 예의주시하고 있었다. 그러다가 그해 10월 13일 박영효, 김옥균 일행이 수신사(修信使) 겸 사죄사(辭罪使)로 하는 사절단(使節團)이 파견되자, 그는 박영효와 김옥균 일행이 도쿄에 방문했을 때 사절단의 통역을 맡아


In [9]:
# data/golden_set.json 저장
GOLDEN_PATH = Path("../data/golden_set.json")
GOLDEN_PATH.write_text(
    json.dumps(golden_set, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# 검증 — 개수 / 고유 질문 수 / 빈 값 없음
loaded = json.loads(GOLDEN_PATH.read_text(encoding="utf-8"))
assert len(loaded) == GOLDEN_SIZE
assert all(d["question"] and d["ground_truth"] and d["reference_contexts"] for d in loaded)

print(f"{len(loaded)}개 골든셋을 {GOLDEN_PATH} 에 저장했습니다.")
print(f"고유 질문: {len({d['question'] for d in loaded})} / {GOLDEN_SIZE}")

50개 골든셋을 ../data/golden_set.json 에 저장했습니다.
고유 질문: 50 / 50
